# Week 10 Lecture: From Numbers to Categories

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bradleyboehmke/uc-bana-4080/blob/main/notebooks/tuesday-your-turn/week-10-lecture.ipynb)

This week the target is a **category** instead of a number. You'll build a logistic regression model that predicts whether a credit card customer will default, turn its output into probabilities, and then find out why 97% accuracy can still be a bad model.

## How to use this notebook

This notebook accompanies the Week 10 Tuesday lecture. You can:

- **Follow along during class**: run each cell as the instructor discusses it
- **Work the Your Turn exercises after class**: they're practice for Thursday's lab
- **Reference after class**: use it alongside the textbook chapters

## This Week's Topics

- **Chapter 23: Logistic Regression**: model a yes/no outcome with an S-shaped probability curve
- **Chapter 24: Evaluating Classification Models**: confusion matrices, precision, recall, and choosing a threshold

## Setup

Today's data comes from the `ISLP` package, which isn't installed in Colab by default. Run this cell first.

> ⏳ The install takes about a minute. If Colab asks you to **restart the session**, click **Restart**, then continue with the next cell (you don't need to rerun this one).

In [ ]:
%pip install ISLP --quiet

Now load the libraries and the `Default` data: 10,000 credit card customers, their balance and income, and whether they defaulted.

In [ ]:
import pandas as pd
import numpy as np
from ISLP import load_data
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score

Default = load_data("Default")

Default.head()

Before building anything, check how common the outcome is:

In [ ]:
Default["default"].value_counts(normalize=True)

**Only about 3% of customers default.** Keep this number in mind; it matters a lot in Section 3.

---

## 1. Build a Logistic Regression Model

Same workflow as last week: set up `X` and `y`, split, then fit on the training set only. Two differences:

- The target must be numeric, so we convert `"Yes"`/`"No"` to `1`/`0`
- `stratify=y` keeps the ~3% default rate the same in the training and test sets

In [ ]:
X = Default[["balance", "income"]]
y = (Default["default"] == "Yes").astype(int)   # 1 = defaulted, 0 = did not

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

model = LogisticRegression()
model.fit(X_train, y_train)

print(f"Intercept:           {model.intercept_[0]:.4f}")
print(f"Balance coefficient: {model.coef_[0][0]:.6f}")
print(f"Income coefficient:  {model.coef_[0][1]:.6f}")

### Reading the coefficients

- **Sign = direction.** Both are positive, so a higher balance or income means higher default risk.
- **Size = how far the log-odds move.** Each coefficient is the change in log-odds for **$1**. Multiply by a step that makes sense for the business:

In [ ]:
steps = {"balance": 1000, "income": 10000}

for feature, coef in zip(X.columns, model.coef_[0]):
    step = steps[feature]
    print(f"+${step:,} {feature}: {coef:.6f} × {step:,} = {coef * step:+.2f} log-odds")

Balance moves the log-odds about 30× more than income, even with a 10× smaller step. In practice, read coefficients for **direction** and **rough magnitude**, then convert to probabilities to explain the results.

### From coefficients to probabilities

`predict_proba()` returns two columns: P(no default) and P(default). `[:, 1]` grabs the probability of **default**. Pass it a DataFrame with the **same column names** used in training.

In [ ]:
customers = pd.DataFrame({
    "balance": [1000, 1500, 2000],
    "income":  [40000, 40000, 40000]
})

customers.assign(prob_default=model.predict_proba(customers)[:, 1].round(3))

Same $500 step in balance, very different jumps in probability. That's the S-curve at work.

### 🎯 Your Turn: Score Some Customers

1. Predict the probability of default for a customer with a **$1,800 balance** and a **$20,000 income**, and for one with the same balance but an **$80,000 income**. How much does income change the probability?
2. Now keep income at $40,000 and find (roughly) the **balance where the probability crosses 50%**. *Hint: try a few balances between $1,500 and $2,500.*

In [ ]:
# Your code here

---

## 2. How Good Is Our Model?

`predict()` turns each probability into a class (0 or 1) using a **50% threshold**. Let's check accuracy on the test set:

In [ ]:
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy:.1%}")

97% sounds great. But compare it to a **lazy model** that ignores the data and predicts "No default" for everyone:

In [ ]:
lazy_pred = np.zeros(len(y_test))   # predict 0 ("No default") for everyone

print(f"Lazy model accuracy: {accuracy_score(y_test, lazy_pred):.1%}")
print(f"Our model accuracy:  {accuracy:.1%}")

Our model barely beats doing nothing. When one class is rare, accuracy hides how badly the model handles the class we care about.

### The confusion matrix

The confusion matrix separates the two kinds of mistakes:

In [ ]:
cm = confusion_matrix(y_test, y_pred)

pd.DataFrame(
    cm,
    index=["Actual: No Default", "Actual: Default"],
    columns=["Predicted: No Default", "Predicted: Default"]
)

Read the bottom row: of the 100 customers who actually defaulted, how many did the model catch?

### Precision and recall

| Metric | Formula | Asks |
|--------|---------|------|
| **Precision** | TP / (TP + FP) | When the model flags a customer, how often is it right? |
| **Recall** | TP / (TP + FN) | Of the customers who default, how many does it catch? |

Like last week's metric functions, these take the **actual values first**, then the **predictions**.

In [ ]:
print(f"Precision: {precision_score(y_test, y_pred):.2f}")
print(f"Recall:    {recall_score(y_test, y_pred):.2f}")

---

## 3. The Threshold Is a Dial

Nothing forces us to use 50%. Get the probabilities, then apply our own threshold:

In [ ]:
probs = model.predict_proba(X_test)[:, 1]

results = []
for threshold in [0.3, 0.4, 0.5, 0.6, 0.7]:
    preds = (probs >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, preds).ravel()
    results.append({
        "threshold": threshold,
        "caught (TP)": tp,
        "missed (FN)": fn,
        "false alarms (FP)": fp,
        "accuracy": accuracy_score(y_test, preds),
        "precision": precision_score(y_test, preds),
        "recall": recall_score(y_test, preds)
    })

pd.DataFrame(results).round(3)

**Lower the threshold** → catch more defaulters (recall ↑) but flag more good customers (precision ↓). Notice that accuracy barely moves. The business's error costs decide where to set the dial.

### 🎯 Your Turn: Pick the Threshold

The bank estimates that a **missed defaulter (FN) costs $5,000** and a **false alarm (FP) costs $200**.

1. Using the thresholds **0.5, 0.3, 0.2, and 0.1**, compute the total cost for each: `fn * 5000 + fp * 200`. Which threshold costs the bank the least?
2. Repeat with a false alarm costing **$2,000**. Does the best threshold change?
3. In a sentence or two: why isn't 50% automatically the right threshold?

*Hint: copy the loop above and add a `"total cost"` entry to the dictionary.*

In [ ]:
# Your code here

*Your answer to question 3:*

---

## 🧾 What You Learned

| Code | What it does |
|------|--------------|
| `y = (df["col"] == "Yes").astype(int)` | Convert a Yes/No target to 1/0 |
| `train_test_split(..., stratify=y)` | Keep the class balance the same in train and test |
| `LogisticRegression().fit(X, y)` | Fit a logistic regression model |
| `model.coef_`, `model.intercept_` | Coefficients (change in log-odds per unit of each feature) |
| `model.predict_proba(X)[:, 1]` | Probability of the positive class (1) |
| `model.predict(X)` | Predicted class using a 50% threshold |
| `(probs >= threshold).astype(int)` | Predicted class using your own threshold |
| `accuracy_score(y_true, y_pred)` | Share of predictions that are correct |
| `confusion_matrix(y_true, y_pred)` | Counts of TN, FP, FN, TP |
| `precision_score(y_true, y_pred)` | When we flag, how often we're right |
| `recall_score(y_true, y_pred)` | Of the actual positives, how many we catch |

## What's Next

- **Thursday Lab:** hands-on classification with medical data. This week's lab is also your homework.
- **Chapters to review:** [Chapter 23](../../book/23-logistic-regression.html) and [Chapter 24](../../book/24-classification-evaluation.html) for deeper coverage, including F1-score, ROC curves, and AUC
- **Cheat sheet:** [Module 10 Cheat Sheet](../../book/module-10-cheatsheet.html) for a quick reference on all of today's concepts